# Training an ML Model for Docker

This notebook trains a simple classifier on the Iris dataset and saves it for the Flask service to use.

## Step 1: Import Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os

## Step 2: Load and Explore the Iris Dataset

In [2]:
# Load the Iris dataset
iris = load_iris()
X = iris.data  # Features: sepal length, sepal width, petal length, petal width
y = iris.target  # Target: iris species (0=setosa, 1=versicolor, 2=virginica)

# Create a DataFrame for easier inspection
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nClass distribution:")
print(df['species'].value_counts())

Dataset shape: (150, 6)

First few rows:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target species  
0       0  setosa  
1       0  setosa  
2       0  setosa  
3       0  setosa  
4       0  setosa  

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   

## Step 3: Split Data into Training and Testing Sets

In [3]:
# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Training set size: 120
Testing set size: 30


## Step 4: Train the Model

In [4]:
# Create and train a Random Forest classifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)

print("Training the model...")
model.fit(X_train, y_train)
print("Model training complete!")

Training the model...
Model training complete!


## Step 5: Evaluate the Model

In [5]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Model Accuracy: 0.9000

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



## Step 6: Save the Model

In [6]:
# Create directory if it doesn't exist
model_dir = '/workspace/Docker/models'
os.makedirs(model_dir, exist_ok=True)

# Save the model
model_path = os.path.join(model_dir, 'iris_model.joblib')
joblib.dump(model, model_path)

print(f"Model saved to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / 1024:.2f} KB")

Model saved to: /workspace/Docker/models/iris_model.joblib
File size: 163.77 KB


## Step 7: Test Loading the Model

In [7]:
# Verify we can load the model back
loaded_model = joblib.load(model_path)

# Test prediction with sample data
sample_data = np.array([[5.1, 3.5, 1.4, 0.2]])  # A setosa iris
prediction = loaded_model.predict(sample_data)
probabilities = loaded_model.predict_proba(sample_data)

species_map = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
print(f"Sample data: {sample_data[0]}")
print(f"Predicted species: {species_map[prediction[0]]}")
print(f"Prediction probabilities: {probabilities[0]}")

Sample data: [5.1 3.5 1.4 0.2]
Predicted species: setosa
Prediction probabilities: [1. 0. 0.]
